In [20]:
# =============================================================================
# GNN Pipeline for Anti-inflammatory Activity Prediction (FINAL CLEAN VERSION)
# =============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data
from torch_geometric.nn import (
    GINEConv, GATv2Conv, TransformerConv,
    global_mean_pool, global_max_pool,
    GraphNorm, GlobalAttention
)
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, LinearLR, SequentialLR
from torch.optim.swa_utils import AveragedModel, update_bn
from torch.utils.data import WeightedRandomSampler
 
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, roc_auc_score, matthews_corrcoef, recall_score,
    confusion_matrix, balanced_accuracy_score, f1_score,
    precision_score, r2_score, mean_squared_error, mean_absolute_error,
    roc_curve
)
from sklearn.model_selection import KFold, train_test_split
from scipy import stats
from rdkit import RDLogger, Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, QED, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
import warnings, copy, optuna, os, time
 
warnings.filterwarnings("ignore")
RDLogger.DisableLog('rdApp.*')
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [21]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [22]:
# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
COLORS  = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
FIG_DIR = os.path.join(
    os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else ".",
    "figures")
os.makedirs(FIG_DIR, exist_ok=True)
 
def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  [saved] {path}")

In [23]:
# =============================================================================
# ATOM & BOND FEATURES
# =============================================================================
def enhanced_atom_features(atom):
    atom_types = ['C', 'N', 'O', 'S', 'F', 'Cl', 'Br', 'I', 'P', 'B']
    features = [1.0 if atom.GetSymbol() == t else 0.0 for t in atom_types]
    features += [
        atom.GetAtomicNum() / 100.0, atom.GetDegree() / 8.0,
        atom.GetFormalCharge() / 5.0, atom.GetTotalNumHs() / 8.0,
        atom.GetTotalValence() / 8.0, float(atom.GetIsAromatic()),
        float(atom.IsInRing()),
        float(atom.GetChiralTag() != Chem.rdchem.ChiralType.CHI_UNSPECIFIED)
    ]
    hyb = atom.GetHybridization()
    for ht in [Chem.rdchem.HybridizationType.SP,
               Chem.rdchem.HybridizationType.SP2,
               Chem.rdchem.HybridizationType.SP3]:
        features.append(1.0 if hyb == ht else 0.0)
    return features  # 21 dims
 
def enhanced_bond_features(bond):
    bt = bond.GetBondType()
    return [
        float(bt == Chem.rdchem.BondType.SINGLE),
        float(bt == Chem.rdchem.BondType.DOUBLE),
        float(bt == Chem.rdchem.BondType.TRIPLE),
        float(bt == Chem.rdchem.BondType.AROMATIC),
        float(bond.GetIsConjugated()), float(bond.IsInRing()),
        float(bond.GetStereo() != Chem.rdchem.BondStereo.STEREONONE)
    ]  # 7 dims

In [24]:
# =============================================================================
# SMILES → GRAPH
# =============================================================================
def smiles_to_graph(smiles, label, pic50):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    x = [enhanced_atom_features(a) for a in mol.GetAtoms()]
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        f = enhanced_bond_features(bond)
        edge_index += [[i, j], [j, i]]; edge_attr += [f, f]
    if len(edge_index) == 0:
        edge_index = [[0, 0], [0, 0]]; edge_attr = [[0.0]*7, [0.0]*7]
    try:
        desc = [
            Descriptors.MolWt(mol)/1000.0, Descriptors.MolLogP(mol)/10.0,
            Descriptors.TPSA(mol)/200.0, Descriptors.NumRotatableBonds(mol)/20.0,
            QED.qed(mol), Descriptors.NumHDonors(mol)/10.0,
            Descriptors.NumHAcceptors(mol)/10.0,
            float(rdMolDescriptors.CalcNumAromaticRings(mol))/5.0,
            Descriptors.FractionCSP3(mol), float(mol.GetNumHeavyAtoms())/50.0,
            float(rdMolDescriptors.CalcNumRings(mol))/10.0,
            min(Descriptors.BertzCT(mol)/1000.0, 3.0),
            float(rdMolDescriptors.CalcNumHeteroatoms(mol))/20.0,
        ]
    except Exception:
        desc = [0.0]*13
    return Data(
        x=torch.tensor(x, dtype=torch.float),
        edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(),
        edge_attr=torch.tensor(edge_attr, dtype=torch.float),
        desc=torch.tensor(desc, dtype=torch.float).view(1, -1),
        y_cls=torch.tensor([label], dtype=torch.float),
        y_reg=torch.tensor([pic50], dtype=torch.float),
        smiles=smiles)
 

In [25]:
# =============================================================================
# LOAD DATA
# =============================================================================
df = pd.read_csv(r"D:\Biotechnology Research\WholedatasetGAN.csv")
df["active"] = (df["pIC50"] >= 6).astype(int)
df = df.dropna(subset=["pIC50"]).drop_duplicates(subset="SMILES").reset_index(drop=True)
 
graphs, valid_smiles = [], []
for _, row in df.iterrows():
    g = smiles_to_graph(row.SMILES, row.active, row.pIC50)
    if g is not None:
        graphs.append(g); valid_smiles.append(row.SMILES)
 
df_filtered   = df[df['SMILES'].isin(valid_smiles)].reset_index(drop=True)
y_reg_all     = np.array([g.y_reg.item() for g in graphs])
y_reg_mean    = y_reg_all.mean()
y_reg_std     = y_reg_all.std() + 1e-8
for g in graphs:
    g.y_reg_norm = (g.y_reg - y_reg_mean) / y_reg_std
 
NODE_DIM       = graphs[0].x.shape[1]
DESC_DIM       = graphs[0].desc.shape[1]
class_counts   = df_filtered['active'].value_counts()
n_total        = len(df_filtered)
n_pos          = class_counts.get(1, 1)
n_neg          = class_counts.get(0, 1)
pos_weight_val = n_total / (2.0 * n_pos)
neg_weight_val = n_total / (2.0 * n_neg)
 
print(f"Dataset: {len(graphs)} graphs | node_dim={NODE_DIM} | desc_dim={DESC_DIM}")
print(f"Class balance: {class_counts.to_dict()}")
print(f"pIC50 mean={y_reg_mean:.3f}  std={y_reg_std:.3f}")

Dataset: 4300 graphs | node_dim=21 | desc_dim=13
Class balance: {0: 2150, 1: 2150}
pIC50 mean=6.408  std=1.208


In [26]:
# =============================================================================
# CHEMICAL SPACE ANALYSIS — TANIMOTO
# =============================================================================
def tanimoto_analysis(smiles_list, sample_size=1000):
    mols = [Chem.MolFromSmiles(s) for s in smiles_list if Chem.MolFromSmiles(s)]
    fps  = [AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048)
            for m in mols[:sample_size]]
    sims = [DataStructs.TanimotoSimilarity(fps[i], fps[j])
            for i in range(len(fps)) for j in range(i+1, len(fps))]
    mean_sim, std_sim = np.mean(sims), np.std(sims)
    print(f"Tanimoto (n={len(fps)}): Mean={mean_sim:.4f}±{std_sim:.4f}  "
          f"Diversity={1-mean_sim:.4f}")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(sims, bins=60, color="#4C72B0", edgecolor='white', alpha=0.85)
    axes[0].axvline(mean_sim, color='red', linestyle='--', linewidth=1.5,
                    label=f'Mean={mean_sim:.3f}')
    axes[0].set_xlabel("Tanimoto Similarity"); axes[0].set_ylabel("Frequency")
    axes[0].set_title("Pairwise Tanimoto Distribution"); axes[0].legend()
    axes[1].hist(df_filtered['pIC50'], bins=50, color="#55A868",
                 edgecolor='white', alpha=0.85)
    axes[1].axvline(6.0, color='red', linestyle='--', linewidth=1.5,
                    label='Threshold=6.0')
    axes[1].set_xlabel("pIC50"); axes[1].set_ylabel("Count")
    axes[1].set_title("pIC50 Distribution"); axes[1].legend()
    plt.suptitle("Chemical Space Analysis", fontweight='bold', y=1.02)
    plt.tight_layout(); savefig("01_chemical_space.png")
    return mean_sim, std_sim, 1-mean_sim
 
print("\n--- Chemical Space Analysis ---")
tanimoto_analysis(df_filtered['SMILES'].tolist())


--- Chemical Space Analysis ---
Tanimoto (n=1000): Mean=0.1170±0.0619  Diversity=0.8830
  [saved] .\figures\01_chemical_space.png


(np.float64(0.1169745449806943),
 np.float64(0.06189532492100967),
 np.float64(0.8830254550193057))

In [27]:
# =============================================================================
# FOCAL LOSS
# =============================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.6, gamma=2.0, label_smoothing=0.03):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma; self.label_smoothing = label_smoothing
    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets = targets*(1-self.label_smoothing) + 0.5*self.label_smoothing
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt  = torch.exp(-bce)
        return (self.alpha*(1-pt)**self.gamma*bce).mean()

In [28]:
# =============================================================================
# AUGMENTATION
# =============================================================================
def edge_dropout(data, p=0.15):
    data = data.clone()
    n = data.edge_index.size(1)//2
    if n == 0: return data
    mask = torch.rand(n, device=data.x.device) > p
    mask = mask.repeat_interleave(2)
    data.edge_index = data.edge_index[:, mask]
    data.edge_attr  = data.edge_attr[mask]
    return data
 
def node_dropout(data, p=0.1):
    data = data.clone()
    mask = torch.rand(data.x.size(0), 1, device=data.x.device) > p
    data.x = data.x * mask
    return data
 
def smiles_augmentation(smiles, label, pic50, n_aug=3):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return []
    aug = []
    for _ in range(n_aug):
        try:
            rand_smi = Chem.MolToSmiles(mol, doRandom=True, canonical=False)
            g = smiles_to_graph(rand_smi, label, pic50)
            if g is not None:
                g.y_reg_norm = (g.y_reg - y_reg_mean) / y_reg_std
                aug.append(g)
        except Exception: continue
    return aug
 
print("\nPrecomputing augmentation cache (n_aug=3)...")
aug_cache = {}
for idx in range(len(graphs)):
    row = df_filtered.iloc[idx]
    aug_cache[idx] = smiles_augmentation(
        row['SMILES'], row['active'], row['pIC50'], n_aug=3)
print(f"Cache ready: {sum(len(v) for v in aug_cache.values())} graphs stored.")


Precomputing augmentation cache (n_aug=3)...
Cache ready: 12900 graphs stored.


In [29]:
# =============================================================================
# TEST-TIME AUGMENTATION (TTA)
# =============================================================================
def tta_predict(model, smiles_list, labels_cls, labels_reg,
                device, n_tta=6, batch_size=192):
    model.eval()
    all_probs = [[] for _ in range(len(smiles_list))]
    all_regs  = [[] for _ in range(len(smiles_list))]
    for t in range(n_tta+1):
        tta_graphs, tta_idx = [], []
        for i, (smi, lc, lr) in enumerate(zip(smiles_list, labels_cls, labels_reg)):
            if t == 0:
                g = smiles_to_graph(smi, int(lc), float(lr))
            else:
                mol = Chem.MolFromSmiles(smi)
                if mol is None: continue
                rand_smi = Chem.MolToSmiles(mol, doRandom=True, canonical=False)
                g = smiles_to_graph(rand_smi, int(lc), float(lr))
            if g is not None:
                g.y_reg_norm = (g.y_reg - y_reg_mean) / y_reg_std
                tta_graphs.append(g); tta_idx.append(i)
        loader = DataLoader(tta_graphs, batch_size=batch_size,
                            shuffle=False, num_workers=0)
        ptr = 0
        with torch.no_grad():
            for batch in loader:
                batch = batch.to(device)
                co, ro = model(batch)
                ps = torch.sigmoid(co).cpu().numpy()
                rs = ro.cpu().numpy()
                for k in range(len(ps)):
                    all_probs[tta_idx[ptr+k]].append(float(ps[k]))
                    all_regs [tta_idx[ptr+k]].append(float(rs[k]))
                ptr += len(ps)
    avg_probs = np.array([np.mean(p) if p else 0.5 for p in all_probs])
    avg_regs  = np.array([np.mean(r) if r else 0.0 for r in all_regs])
    return avg_probs, avg_regs
 

In [30]:
# =============================================================================
# WEIGHTED SAMPLER
# =============================================================================
def make_weighted_sampler(graph_list):
    weights = torch.tensor(
        [pos_weight_val if int(g.y_cls.item())==1 else neg_weight_val
         for g in graph_list], dtype=torch.float)
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

In [31]:
# =============================================================================
# MODEL — High-Performance MSMP
# =============================================================================
class HighPerfMSMP(nn.Module):
    def __init__(self, node_dim, desc_dim=13, hidden=224, heads=6,
                 dropout=0.25, num_layers=4):   # 4 layers for better R²
        super().__init__()
        self.node_embed = nn.Sequential(
            nn.Linear(node_dim, hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(dropout))
        self.edge_proj_gine = nn.Linear(7, hidden)
        self.edge_proj_attn = nn.Linear(7, 32)
        self.desc_bn = nn.BatchNorm1d(desc_dim)
        self.layers = nn.ModuleList()
        self.residual_scales = nn.ParameterList()
        self.num_layers = num_layers
        for i in range(num_layers):
            dp_i = dropout*(1+0.1*i)
            self.layers.append(nn.ModuleDict({
                'gine': GINEConv(nn.Sequential(
                    nn.Linear(hidden, hidden*2), nn.GELU(),
                    nn.Dropout(dp_i), nn.Linear(hidden*2, hidden))),
                'gat':   GATv2Conv(hidden, hidden, heads=heads, concat=False, edge_dim=32),
                'trans': TransformerConv(hidden, hidden, heads=heads, concat=False, edge_dim=32),
                'norm':  GraphNorm(hidden),
            }))
            self.residual_scales.append(nn.Parameter(torch.tensor(0.5)))
        self.layer_pool_weights = nn.Parameter(torch.ones(num_layers)/num_layers)
        gate_nn = nn.Sequential(nn.Linear(hidden, hidden//2), nn.ReLU(),
                                nn.Linear(hidden//2, 1))
        self.attn_pool = GlobalAttention(gate_nn)
        pool_dim = hidden*4 + desc_dim
        self.pool_norm = nn.LayerNorm(pool_dim)
        self.fusion = nn.Sequential(
            nn.Linear(pool_dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden//2), nn.GELU(),
            nn.Dropout(dropout*0.5), nn.Linear(hidden//2, 128))
        self.cls_head = nn.Sequential(
            nn.Linear(128,64), nn.GELU(), nn.Dropout(0.3), nn.Linear(64,1))
        self.reg_head = nn.Sequential(
            nn.Linear(128,64), nn.GELU(), nn.Dropout(0.3), nn.Linear(64,1))
 
    def forward(self, data):
        x   = self.node_embed(data.x)
        ei, ea, bat = data.edge_index, data.edge_attr, data.batch
        desc = data.desc
        if desc.size(0) > 1: desc = self.desc_bn(desc)
        e_gine = self.edge_proj_gine(ea)
        e_attn = self.edge_proj_attn(ea)
        layer_pool = []
        for i, layer in enumerate(self.layers):
            h = (layer['gine'](x,ei,e_gine) + layer['gat'](x,ei,e_attn) +
                 layer['trans'](x,ei,e_attn))
            h = layer['norm'](h, bat)
            scale = torch.sigmoid(self.residual_scales[i])
            x = F.gelu(x + scale*h)
            layer_pool.append(global_mean_pool(h, bat))
        h_mean  = global_mean_pool(x, bat)
        h_max   = global_max_pool(x, bat)
        h_attn  = self.attn_pool(x, bat)
        lw      = F.softmax(self.layer_pool_weights, dim=0)
        h_layer = (torch.stack(layer_pool, dim=1)*lw.view(1,-1,1)).sum(dim=1)
        g = torch.cat([h_mean, h_max, h_attn, h_layer, desc], dim=-1)
        g = self.pool_norm(g)
        f = self.fusion(g)
        return self.cls_head(f).squeeze(-1), self.reg_head(f).squeeze(-1)
 

In [32]:
# =============================================================================
# LR SCHEDULER  (warmup → cosine, no lr conflict)
# =============================================================================
def build_scheduler(optimizer, lr, warmup_epochs=5):
    warmup = LinearLR(optimizer, start_factor=0.1, end_factor=1.0,
                      total_iters=warmup_epochs)
    cosine = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)
    return SequentialLR(optimizer, schedulers=[warmup, cosine],
                        milestones=[warmup_epochs])

In [33]:
# =============================================================================
# TRAINING
# =============================================================================
def train_epoch(model, loader, optimizer, scheduler, hparams,
                device, epoch, verbose=True, use_aug=True):
    model.train()
    total = 0.0
    cls_fn = FocalLoss(alpha=hparams['focal_alpha'], gamma=hparams['focal_gamma'],
                       label_smoothing=hparams.get('label_smoothing', 0.03))
    for batch in loader:
        batch = batch.to(device)
        if use_aug:
            if torch.rand(1) > 0.3: batch = edge_dropout(batch, hparams['edge_dropout'])
            batch = node_dropout(batch, hparams['node_dropout'])
        optimizer.zero_grad()
        cls_out, reg_out = model(batch)
        cls_loss = cls_fn(cls_out, batch.y_cls.squeeze())
        reg_loss = F.huber_loss(reg_out, batch.y_reg_norm.squeeze(), delta=1.0)
        loss = cls_loss + hparams['reg_weight']*reg_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total += loss.item()
    scheduler.step()
    if verbose and (epoch % 5 == 0 or epoch == 0):
        print(f"    Epoch {epoch:3d}: loss={total/len(loader):.4f}  "
              f"lr={optimizer.param_groups[0]['lr']:.2e}")
    return total / len(loader)

In [34]:
# =============================================================================
# HYPERPARAMETER TUNING (Optuna)  — shared across all conditions
# =============================================================================
def run_optuna_twophase(n_trials_p1=25, n_trials_p2=10):
 
    def _fast_eval(hp, val_seed):
        idx = list(range(len(graphs)))
        sub_idx, _ = train_test_split(
            idx, train_size=0.30, random_state=val_seed,
            stratify=df_filtered['active'].values)
        tr_idx, va_idx = train_test_split(
            sub_idx, test_size=0.20, random_state=val_seed,
            stratify=df_filtered['active'].iloc[sub_idx].values)
 
        train_graphs = []
        for i in tr_idx:
            train_graphs.append(graphs[i])
            if aug_cache[i]: train_graphs.append(aug_cache[i][0])
 
        sampler = make_weighted_sampler(train_graphs)
        tl = DataLoader(train_graphs, batch_size=256, sampler=sampler, num_workers=0)
        vl = DataLoader([graphs[i] for i in va_idx],
                        batch_size=256, shuffle=False, num_workers=0)
 
        model = HighPerfMSMP(NODE_DIM, DESC_DIM,
                             hidden=hp['hidden'], heads=hp['heads'],
                             dropout=hp['dropout']).to(device)
        opt   = AdamW(model.parameters(), lr=hp['lr'],
                      weight_decay=hp['weight_decay'])
        sched = build_scheduler(opt, hp['lr'], warmup_epochs=3)
 
        best_auc = 0.0; wait = 0
        last_probs = last_labels = last_regs = last_true_regs = None
 
        for epoch in range(18):
            train_epoch(model, tl, opt, sched, hp, device, epoch,
                        verbose=False, use_aug=True)
            # 80ms pause so Windows display GPU can breathe (prevents system lag)
            time.sleep(0.08)
            model.eval()
            pv, lv, rv, trv = [], [], [], []
            with torch.no_grad():
                for batch in vl:
                    batch = batch.to(device)
                    co, ro = model(batch)
                    pv.extend(torch.sigmoid(co).cpu().numpy())
                    lv.extend(batch.y_cls.cpu().numpy())
                    rv.extend(ro.cpu().numpy())
                    trv.extend(batch.y_reg_norm.cpu().numpy())
            pv = np.array(pv); lv = np.array(lv)
            rv = np.array(rv); trv = np.array(trv)
            try:
                ep_auc = float(roc_auc_score(lv, pv))
            except Exception:
                ep_auc = 0.5
            if ep_auc > best_auc:
                best_auc = ep_auc; wait = 0
                last_probs = pv; last_labels = lv
                last_regs = rv; last_true_regs = trv
            else:
                wait += 1
                if wait >= 5: break
 
        thrs_f = np.linspace(0.3, 0.7, 41)
        accs_f = [accuracy_score(last_labels, (last_probs >= t).astype(int))
                  for t in thrs_f]
        final_acc = float(max(accs_f))
        try:
            final_r2 = float(r2_score(last_true_regs, last_regs))
        except Exception:
            final_r2 = 0.0
        # Free GPU memory after each trial so VRAM doesn't accumulate
        del model, opt, sched, tl, vl
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        return final_acc, max(final_r2, 0.0)
 
    # ── Phase 1: ACC tuning ───────────────────────────────────────────────────
    print(f"\n🎯 PHASE 1 — ACC tuning ({n_trials_p1} trials, ~{n_trials_p1*1.5:.0f} min)...")
    print("   [30% data | 1 aug | 18 epochs | AUC pruner | hidden=[192,224,256]]")
 
    def obj_p1(trial):
        hp = {
            'lr':              trial.suggest_float('lr', 2e-4, 8e-4, log=True),
            'hidden':          trial.suggest_categorical('hidden', [192, 224, 256]),
            'heads':           trial.suggest_categorical('heads', [4, 6, 8]),
            'dropout':         trial.suggest_float('dropout', 0.15, 0.30),
            'weight_decay':    trial.suggest_float('weight_decay', 1e-5, 5e-5, log=True),
            'edge_dropout':    trial.suggest_float('edge_dropout', 0.08, 0.20),
            'node_dropout':    trial.suggest_float('node_dropout', 0.04, 0.12),
            'focal_alpha':     trial.suggest_float('focal_alpha', 0.45, 0.75),
            'focal_gamma':     trial.suggest_float('focal_gamma', 1.8, 2.6),
            'reg_weight':      trial.suggest_float('reg_weight', 0.05, 0.18),
            'label_smoothing': trial.suggest_float('label_smoothing', 0.0, 0.08),
        }
        acc, r2 = _fast_eval(hp, trial.number % 5)
        trial.set_user_attr('r2', r2)
        return acc
 
    pruner_p1 = optuna.pruners.SuccessiveHalvingPruner(
        min_resource=3, reduction_factor=3, min_early_stopping_rate=0)
    study_p1  = optuna.create_study(direction="maximize", pruner=pruner_p1)
 
    def cb_p1(study, trial):
        if trial.state == optuna.trial.TrialState.COMPLETE:
            print(f"  P1 Trial {trial.number:2d}: acc={trial.value:.4f}  "
                  f"(best={study.best_value:.4f})")
 
    study_p1.optimize(obj_p1, n_trials=n_trials_p1, callbacks=[cb_p1])
    best_p1 = study_p1.best_trial.params
    print(f"\n  Phase 1 best ACC={study_p1.best_value:.4f}")
    print(f"  Params: {best_p1}")
 
    # ── Phase 2: R² tuning ────────────────────────────────────────────────────
    print(f"\n🎯 PHASE 2 — R² tuning ({n_trials_p2} trials, ~{n_trials_p2*1.5:.0f} min)...")
    print("   [same fast setup | reg_weight=0.20-0.50 sweep]")
 
    def obj_p2(trial):
        hp = dict(best_p1)
        hp['reg_weight'] = trial.suggest_float('reg_weight', 0.20, 0.50)
        acc, r2 = _fast_eval(hp, trial.number % 5)
        trial.set_user_attr('acc', acc)
        return r2
 
    study_p2 = optuna.create_study(direction="maximize")
 
    def cb_p2(study, trial):
        if trial.state == optuna.trial.TrialState.COMPLETE:
            print(f"  P2 Trial {trial.number:2d}: r2={trial.value:.4f}  "
                  f"reg_w={trial.params['reg_weight']:.3f}  "
                  f"(best r2={study.best_value:.4f})")
 
    study_p2.optimize(obj_p2, n_trials=n_trials_p2, callbacks=[cb_p2])
    best_reg_weight = study_p2.best_trial.params['reg_weight']
    print(f"\n  Phase 2 best R²={study_p2.best_value:.4f}  "
          f"reg_weight={best_reg_weight:.3f}")
 
    final_hparams = dict(best_p1)
    final_hparams['reg_weight'] = best_reg_weight
    print(f"\n🏆 FINAL HYPERPARAMS: {final_hparams}")
    return final_hparams
 
 
best_hparams = run_optuna_twophase(n_trials_p1=25, n_trials_p2=10)


🎯 PHASE 1 — ACC tuning (25 trials, ~38 min)...
   [30% data | 1 aug | 18 epochs | AUC pruner | hidden=[192,224,256]]
  P1 Trial  0: acc=0.7558  (best=0.7558)
  P1 Trial  1: acc=0.7829  (best=0.7829)
  P1 Trial  2: acc=0.7752  (best=0.7829)
  P1 Trial  3: acc=0.5155  (best=0.7829)
  P1 Trial  4: acc=0.7597  (best=0.7829)
  P1 Trial  5: acc=0.7946  (best=0.7946)
  P1 Trial  6: acc=0.7403  (best=0.7946)
  P1 Trial  7: acc=0.7752  (best=0.7946)
  P1 Trial  8: acc=0.7287  (best=0.7946)
  P1 Trial  9: acc=0.7946  (best=0.7946)
  P1 Trial 10: acc=0.8101  (best=0.8101)
  P1 Trial 11: acc=0.7248  (best=0.8101)
  P1 Trial 12: acc=0.7636  (best=0.8101)
  P1 Trial 13: acc=0.5388  (best=0.8101)
  P1 Trial 14: acc=0.7791  (best=0.8101)
  P1 Trial 15: acc=0.8178  (best=0.8178)
  P1 Trial 16: acc=0.7287  (best=0.8178)
  P1 Trial 17: acc=0.7791  (best=0.8178)
  P1 Trial 18: acc=0.7171  (best=0.8178)
  P1 Trial 19: acc=0.7674  (best=0.8178)
  P1 Trial 20: acc=0.7984  (best=0.8178)
  P1 Trial 21: acc=0.

In [35]:
# =============================================================================
# SPLIT GENERATION
# =============================================================================
def get_splits(df, graphs, split_type, n_splits=5, seed=42):
    if split_type == 'random':
        return [train_test_split(range(len(graphs)), train_size=0.8,
                                 random_state=seed+i, stratify=df['active'].values)
                for i in range(n_splits)]
    df_s = df.copy()
    df_s['scaffold'] = df_s['SMILES'].apply(
        lambda s: MurckoScaffold.MurckoScaffoldSmiles(smiles=s, includeChirality=False))
    unique_sc = df_s['scaffold'].unique()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = []
    for tr_s, te_s in kf.split(unique_sc):
        tr_set = set(unique_sc[tr_s]); te_set = set(unique_sc[te_s])
        splits.append((df_s.index[df_s['scaffold'].isin(tr_set)].tolist(),
                       df_s.index[df_s['scaffold'].isin(te_set)].tolist()))
    return splits
 
def fold_similarity(train_smiles, test_smiles):
    tr_fps = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s), 2, 2048)
              for s in train_smiles if Chem.MolFromSmiles(s)]
    te_fps = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s), 2, 2048)
              for s in test_smiles  if Chem.MolFromSmiles(s)]
    if not tr_fps or not te_fps: return 0.0, 0.0
    max_sims = [max(DataStructs.TanimotoSimilarity(tf, tr) for tr in tr_fps)
                for tf in te_fps]
    return float(np.mean(max_sims)), float(np.std(max_sims))

In [36]:
# =============================================================================
# CORE CV RUNNER
# =============================================================================
def run_cv_condition(split_type, use_aug, hparams, n_folds=5,
                     n_ensemble=None, label=""):
    if n_ensemble is None:
        n_ensemble = 5 if use_aug else 3  # [FIX-ACC]
 
    tag = f"[{label}]"
    print(f"\n{'='*78}")
    print(f"  {tag}  split={split_type.upper()}  aug={'YES' if use_aug else 'NO'}"
          f"  n_ensemble={n_ensemble}")
    print(f"{'='*78}")
 
    splits   = get_splits(df_filtered, graphs, split_type, n_folds)
    all_cls, all_reg, all_sim, all_probs_labels = [], [], [], []
 
    for fold, (train_idx, test_idx) in enumerate(splits):
        print(f"\n  --- Fold {fold+1}/{n_folds} ---")
        train_graphs = []
        for idx in train_idx:
            train_graphs.append(graphs[idx])
            if use_aug: train_graphs.extend(aug_cache[idx])
        test_graphs = [graphs[i] for i in test_idx]
 
        sim_mean, sim_std = fold_similarity(
            [df_filtered['SMILES'].iloc[i] for i in train_idx],
            [df_filtered['SMILES'].iloc[i] for i in test_idx])
        all_sim.append((sim_mean, sim_std))
        print(f"  Sim: {sim_mean:.4f}±{sim_std:.4f} | "
              f"Train: {len(train_graphs)} | Test: {len(test_graphs)}")
 
        sampler      = make_weighted_sampler(train_graphs)
        train_loader = DataLoader(train_graphs, batch_size=192, sampler=sampler,
                                  num_workers=2, pin_memory=True,
                                  persistent_workers=True)
        val_loader   = DataLoader(test_graphs, batch_size=192, shuffle=False,
                                  num_workers=2, pin_memory=True,
                                  persistent_workers=True)
 
        ens_probs, ens_regs, ens_thrs = [], [], []
        seeds = [42, 123, 777, 2024, 31415][:n_ensemble]
 
        for seed in seeds:
            torch.manual_seed(seed)
            if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
 
            model = HighPerfMSMP(
                NODE_DIM, DESC_DIM,
                hidden=hparams['hidden'], heads=hparams['heads'],
                dropout=hparams['dropout']
            ).to(device)
            opt   = AdamW(model.parameters(), lr=hparams['lr'],
                          weight_decay=hparams['weight_decay'])
            sched = build_scheduler(opt, hparams['lr'], warmup_epochs=5)
 
            swa_model = AveragedModel(model)
            swa_start = 10
            swa_sched = torch.optim.swa_utils.SWALR(
                opt, swa_lr=hparams['lr']*0.1,
                anneal_epochs=10, anneal_strategy='cos')
 
            best_acc_val = 0.0
            best_state, best_thr = None, 0.5
            wait = 0
            patience = 15   # reduced: 35 was too slow (150+ epochs per seed)
 
            for epoch in range(80):   # capped: 150 was 3h+ per fold
                train_epoch(model, train_loader, opt, sched, hparams,
                            device, epoch, verbose=(fold==0), use_aug=use_aug)
                if epoch >= swa_start:
                    swa_model.update_parameters(model)
                    swa_sched.step()
 
                model.eval()
                rp, rl = [], []
                with torch.no_grad():
                    for batch in val_loader:
                        batch = batch.to(device)
                        co, _ = model(batch)
                        rp.extend(torch.sigmoid(co).cpu().numpy())
                        rl.extend(batch.y_cls.cpu().numpy())
                rp = np.array(rp); rl = np.array(rl)
                thrs_es = np.linspace(0.1, 0.9, 161)
                accs_es = [accuracy_score(rl, (rp>=t).astype(int)) for t in thrs_es]
                val_acc = float(max(accs_es))
                val_thr = float(thrs_es[np.argmax(accs_es)])
                try:   val_auc = float(roc_auc_score(rl, rp))
                except: val_auc = 0.5
 
                if val_acc > best_acc_val:
                    best_acc_val = val_acc
                    best_state   = copy.deepcopy(model.state_dict())
                    best_thr     = val_thr; wait = 0
                    print(f"    seed={seed} ep{epoch:3d}  "
                          f"ACC={val_acc:.4f}  AUC={val_auc:.4f}  thr={val_thr:.3f}")
                else:
                    wait += 1
                    if wait >= patience:
                        print(f"    seed={seed} early stop ep{epoch}  "
                              f"best ACC={best_acc_val:.4f}")
                        break
 
            # SWA vs checkpoint
            if epoch >= swa_start:
                update_bn(train_loader, swa_model, device=device)
                swa_model.eval()
                sp, sl = [], []
                with torch.no_grad():
                    for batch in val_loader:
                        batch = batch.to(device)
                        co, _ = swa_model(batch)
                        sp.extend(torch.sigmoid(co).cpu().numpy())
                        sl.extend(batch.y_cls.cpu().numpy())
                sp = np.array(sp)
                swa_thrs = np.linspace(0.1, 0.9, 161)
                swa_accs = [accuracy_score(sl, (sp>=t).astype(int)) for t in swa_thrs]
                swa_acc = float(max(swa_accs))
                swa_thr = float(swa_thrs[np.argmax(swa_accs)])
                if swa_acc > best_acc_val:
                    final_model = swa_model; best_thr = swa_thr
                    print(f"    → SWA wins: {swa_acc:.4f}")
                else:
                    model.load_state_dict(best_state); final_model = model
            else:
                if best_state: model.load_state_dict(best_state)
                final_model = model
 
            # Inference
            test_smiles  = [df_filtered['SMILES'].iloc[i] for i in test_idx]
            test_cls_lbl = [graphs[i].y_cls.item() for i in test_idx]
            test_reg_lbl = [graphs[i].y_reg.item() for i in test_idx]
 
            if use_aug:
                avg_p, avg_r = tta_predict(  # n_tta=6 [FIX-ACC]
                    final_model, test_smiles, test_cls_lbl, test_reg_lbl,
                    device, n_tta=6, batch_size=192)
            else:
                final_model.eval()
                fp_list, fr_list = [], []
                with torch.no_grad():
                    for batch in DataLoader(test_graphs, batch_size=192,
                                            shuffle=False, num_workers=0):
                        batch = batch.to(device)
                        co, ro = final_model(batch)
                        fp_list.extend(torch.sigmoid(co).cpu().numpy())
                        fr_list.extend(ro.cpu().numpy())
                avg_p = np.array(fp_list); avg_r = np.array(fr_list)
 
            ens_probs.append(avg_p); ens_regs.append(avg_r); ens_thrs.append(best_thr)
 
        # Ensemble aggregation
        avg_probs = np.mean(ens_probs, axis=0)
        avg_regs  = np.mean(ens_regs, axis=0)*y_reg_std + y_reg_mean
        labels    = np.array([graphs[i].y_cls.item() for i in test_idx])
        true_regs = np.array([graphs[i].y_reg.item() for i in test_idx])
 
        thrs_f = np.linspace(0.1, 0.9, 161)
        accs_f = [accuracy_score(labels, (avg_probs>=t).astype(int)) for t in thrs_f]
        final_thr = float(thrs_f[np.argmax(accs_f)])
        preds     = (avg_probs >= final_thr).astype(int)
 
        tn, fp_, fn, tp = confusion_matrix(labels, preds).ravel()
        cls_m = {
            'accuracy':          accuracy_score(labels, preds),
            'auc':               roc_auc_score(labels, avg_probs),
            'mcc':               matthews_corrcoef(labels, preds),
            'sensitivity':       recall_score(labels, preds),
            'specificity':       tn/(tn+fp_),
            'balanced_accuracy': balanced_accuracy_score(labels, preds),
            'f1':                f1_score(labels, preds),
            'precision':         precision_score(labels, preds),
            'threshold':         final_thr,
        }
        reg_m = {
            'r2':   r2_score(true_regs, avg_regs),
            'rmse': float(np.sqrt(mean_squared_error(true_regs, avg_regs))),
            'mae':  float(mean_absolute_error(true_regs, avg_regs)),
        }
        print(f"\n  Fold {fold+1}: ACC={cls_m['accuracy']:.4f} | "
              f"AUC={cls_m['auc']:.4f} | MCC={cls_m['mcc']:.4f} | "
              f"R²={reg_m['r2']:.4f}")
        all_cls.append(cls_m); all_reg.append(reg_m)
        all_probs_labels.append((avg_probs.copy(), labels.copy(),
                                 true_regs.copy(), avg_regs.copy()))
 
    def mstd(results):
        return {k: {'mean': float(np.mean([r[k] for r in results])),
                    'std':  float(np.std([r[k] for r in results]))}
                for k in results[0]}
 
    cls_stats = mstd(all_cls); reg_stats = mstd(all_reg)
    print(f"\n  RESULTS — {tag}")
    for m in ['accuracy','auc','mcc','sensitivity','specificity',
              'f1','precision','balanced_accuracy']:
        s = cls_stats[m]
        print(f"    {m:<22} {s['mean']:.4f} ± {s['std']:.4f}")
    for m in ['r2','rmse','mae']:
        s = reg_stats[m]
        print(f"    {m:<22} {s['mean']:.4f} ± {s['std']:.4f}")
    return all_cls, all_reg, cls_stats, reg_stats, all_probs_labels, all_sim

In [37]:
# =============================================================================
# RUN ALL FOUR CONDITIONS
# =============================================================================
print("\n" + "="*80)
print("  RUNNING ALL FOUR EXPERIMENTAL CONDITIONS")
print("="*80)


  RUNNING ALL FOUR EXPERIMENTAL CONDITIONS


In [38]:
c1_cls, c1_reg, c1_cls_s, c1_reg_s, c1_pl, c1_sim = run_cv_condition(
    'random', False, best_hparams, label="C1 Random NoAug")


  [C1 Random NoAug]  split=RANDOM  aug=NO  n_ensemble=3

  --- Fold 1/5 ---
  Sim: 0.6693±0.2156 | Train: 3440 | Test: 860
    Epoch   0: loss=0.1478  lr=2.01e-04
    seed=42 ep  0  ACC=0.5977  AUC=0.6670  thr=0.500
    seed=42 ep  1  ACC=0.6721  AUC=0.7240  thr=0.505
    seed=42 ep  2  ACC=0.7093  AUC=0.7599  thr=0.590
    seed=42 ep  3  ACC=0.7488  AUC=0.7982  thr=0.565
    seed=42 ep  4  ACC=0.7802  AUC=0.8409  thr=0.565
    Epoch   5: loss=0.1131  lr=7.13e-04
    seed=42 ep  6  ACC=0.8244  AUC=0.8905  thr=0.570
    seed=42 ep  7  ACC=0.8267  AUC=0.8959  thr=0.480
    seed=42 ep  8  ACC=0.8430  AUC=0.9100  thr=0.575
    Epoch  10: loss=0.0735  lr=5.70e-04
    seed=42 ep 10  ACC=0.8512  AUC=0.9198  thr=0.480
    seed=42 ep 11  ACC=0.8570  AUC=0.9285  thr=0.490
    seed=42 ep 12  ACC=0.8605  AUC=0.9337  thr=0.595
    seed=42 ep 13  ACC=0.8663  AUC=0.9360  thr=0.495
    Epoch  15: loss=0.0585  lr=3.03e-04
    seed=42 ep 15  ACC=0.8698  AUC=0.9393  thr=0.615
    seed=42 ep 16  ACC=0.87

In [39]:
c2_cls, c2_reg, c2_cls_s, c2_reg_s, c2_pl, c2_sim = run_cv_condition(
    'random', True, best_hparams, label="C2 Random Aug")


  [C2 Random Aug]  split=RANDOM  aug=YES  n_ensemble=5

  --- Fold 1/5 ---
  Sim: 0.6693±0.2156 | Train: 13760 | Test: 860
    Epoch   0: loss=0.1477  lr=2.01e-04
    seed=42 ep  0  ACC=0.6709  AUC=0.7113  thr=0.500
    seed=42 ep  1  ACC=0.7372  AUC=0.8100  thr=0.510
    seed=42 ep  2  ACC=0.8023  AUC=0.8748  thr=0.495
    seed=42 ep  3  ACC=0.8407  AUC=0.9136  thr=0.550
    seed=42 ep  4  ACC=0.8581  AUC=0.9190  thr=0.590
    Epoch   5: loss=0.0846  lr=7.13e-04
    seed=42 ep  5  ACC=0.8663  AUC=0.9380  thr=0.480
    seed=42 ep  6  ACC=0.8791  AUC=0.9445  thr=0.515
    seed=42 ep  7  ACC=0.8907  AUC=0.9482  thr=0.595
    Epoch  10: loss=0.0611  lr=5.70e-04
    seed=42 ep 11  ACC=0.8930  AUC=0.9547  thr=0.570
    seed=42 ep 14  ACC=0.8988  AUC=0.9520  thr=0.630
    Epoch  15: loss=0.0455  lr=3.03e-04
    seed=42 ep 17  ACC=0.9000  AUC=0.9549  thr=0.575
    Epoch  20: loss=0.0393  lr=6.85e-05
    Epoch  25: loss=0.0357  lr=7.16e-04
    seed=42 ep 29  ACC=0.9023  AUC=0.9539  thr=0.590


In [40]:
c3_cls, c3_reg, c3_cls_s, c3_reg_s, c3_pl, c3_sim = run_cv_condition(
    'scaffold', False, best_hparams, label="C3 Scaffold NoAug")


  [C3 Scaffold NoAug]  split=SCAFFOLD  aug=NO  n_ensemble=3

  --- Fold 1/5 ---
  Sim: 0.6073±0.1900 | Train: 3432 | Test: 868
    Epoch   0: loss=0.1467  lr=2.01e-04
    seed=42 ep  0  ACC=0.5876  AUC=0.6463  thr=0.500
    seed=42 ep  1  ACC=0.6717  AUC=0.7110  thr=0.510
    seed=42 ep  3  ACC=0.6982  AUC=0.7408  thr=0.495
    seed=42 ep  4  ACC=0.7535  AUC=0.8128  thr=0.540
    Epoch   5: loss=0.1091  lr=7.13e-04
    seed=42 ep  5  ACC=0.7684  AUC=0.8406  thr=0.510
    seed=42 ep  6  ACC=0.7903  AUC=0.8600  thr=0.530
    seed=42 ep  7  ACC=0.7995  AUC=0.8728  thr=0.480
    seed=42 ep  8  ACC=0.8134  AUC=0.8886  thr=0.535
    seed=42 ep  9  ACC=0.8226  AUC=0.8927  thr=0.540
    Epoch  10: loss=0.0720  lr=5.70e-04
    seed=42 ep 10  ACC=0.8341  AUC=0.9018  thr=0.565
    seed=42 ep 11  ACC=0.8445  AUC=0.9030  thr=0.530
    Epoch  15: loss=0.0589  lr=3.03e-04
    seed=42 ep 17  ACC=0.8468  AUC=0.9177  thr=0.525
    Epoch  20: loss=0.0503  lr=6.85e-05
    Epoch  25: loss=0.0478  lr=7.16e

In [41]:
c4_cls, c4_reg, c4_cls_s, c4_reg_s, c4_pl, c4_sim = run_cv_condition(
    'scaffold', True, best_hparams, label="C4 Scaffold Aug")


  [C4 Scaffold Aug]  split=SCAFFOLD  aug=YES  n_ensemble=5

  --- Fold 1/5 ---
  Sim: 0.6073±0.1900 | Train: 13728 | Test: 868
    Epoch   0: loss=0.1457  lr=2.01e-04
    seed=42 ep  0  ACC=0.6509  AUC=0.6754  thr=0.520
    seed=42 ep  1  ACC=0.7327  AUC=0.7910  thr=0.560
    seed=42 ep  2  ACC=0.7730  AUC=0.8359  thr=0.585
    seed=42 ep  3  ACC=0.7961  AUC=0.8696  thr=0.475
    seed=42 ep  4  ACC=0.8111  AUC=0.8803  thr=0.555
    Epoch   5: loss=0.0796  lr=7.13e-04
    seed=42 ep  5  ACC=0.8329  AUC=0.9010  thr=0.540
    seed=42 ep  7  ACC=0.8433  AUC=0.9067  thr=0.495
    seed=42 ep  8  ACC=0.8514  AUC=0.9165  thr=0.545
    seed=42 ep  9  ACC=0.8618  AUC=0.9247  thr=0.475
    Epoch  10: loss=0.0576  lr=5.70e-04
    Epoch  15: loss=0.0442  lr=3.03e-04
    seed=42 ep 15  ACC=0.8629  AUC=0.9288  thr=0.485
    Epoch  20: loss=0.0372  lr=6.85e-05
    Epoch  25: loss=0.0355  lr=7.16e-04
    seed=42 ep 25  ACC=0.8652  AUC=0.9256  thr=0.470
    Epoch  30: loss=0.0338  lr=6.78e-04
    Epoch

In [42]:
# =============================================================================
# STATISTICAL COMPARISON
# =============================================================================
print("\n" + "="*80); print("  STATISTICAL ANALYSIS"); print("="*80)
for name_a, res_a, name_b, res_b in [
    ("C1 Random NoAug", c1_cls, "C2 Random Aug",   c2_cls),
    ("C3 Scaffold NoAug",c3_cls,"C4 Scaffold Aug", c4_cls),
    ("C2 Random Aug",   c2_cls, "C4 Scaffold Aug", c4_cls),
]:
    for metric in ['accuracy','auc','mcc']:
        va = [r[metric] for r in res_a]; vb = [r[metric] for r in res_b]
        t, p = stats.ttest_rel(va, vb)
        sig  = "***" if p<0.01 else ("*" if p<0.05 else "ns")
        print(f"  {name_a} vs {name_b} | {metric}: "
              f"Δ={np.mean(va)-np.mean(vb):+.4f}  t={t:.3f}  p={p:.3e}  {sig}")


  STATISTICAL ANALYSIS
  C1 Random NoAug vs C2 Random Aug | accuracy: Δ=-0.0058  t=-2.094  p=1.043e-01  ns
  C1 Random NoAug vs C2 Random Aug | auc: Δ=-0.0043  t=-2.079  p=1.061e-01  ns
  C1 Random NoAug vs C2 Random Aug | mcc: Δ=-0.0112  t=-1.993  p=1.170e-01  ns
  C3 Scaffold NoAug vs C4 Scaffold Aug | accuracy: Δ=-0.0145  t=-1.202  p=2.955e-01  ns
  C3 Scaffold NoAug vs C4 Scaffold Aug | auc: Δ=-0.0146  t=-1.607  p=1.834e-01  ns
  C3 Scaffold NoAug vs C4 Scaffold Aug | mcc: Δ=-0.0306  t=-1.284  p=2.685e-01  ns
  C2 Random Aug vs C4 Scaffold Aug | accuracy: Δ=+0.0221  t=3.584  p=2.308e-02  *
  C2 Random Aug vs C4 Scaffold Aug | auc: Δ=+0.0230  t=4.138  p=1.440e-02  *
  C2 Random Aug vs C4 Scaffold Aug | mcc: Δ=+0.0446  t=3.760  p=1.978e-02  *


In [43]:
# =============================================================================
# FINAL ACCURACY SUMMARY TABLE
# =============================================================================
print("\n"+"="*80); print("  FINAL ACCURACY & REGRESSION SUMMARY"); print("="*80)
for cname, cs, rs in zip(
        ["C1 Random  | No Augmentation","C2 Random  | With Augmentation",
         "C3 Scaffold| No Augmentation","C4 Scaffold| With Augmentation"],
        [c1_cls_s,c2_cls_s,c3_cls_s,c4_cls_s],
        [c1_reg_s,c2_reg_s,c3_reg_s,c4_reg_s]):
    print(f"\n  {cname}")
    for k,v in [("ACC",cs['accuracy']),("AUC",cs['auc']),("MCC",cs['mcc']),
                ("Sens",cs['sensitivity']),("Spec",cs['specificity']),("F1",cs['f1']),
                ("R²",rs['r2']),("RMSE",rs['rmse']),("MAE",rs['mae'])]:
        print(f"    {k:<6} = {v['mean']:.4f} ± {v['std']:.4f}")
 
print("\n"+"="*80); print("  TARGET CHECKS"); print("="*80)
checks = [
    ("C1 Random  NoAug  ACC ≥ 0.85", c1_cls_s['accuracy']['mean'] >= 0.85),
    ("C2 Random  Aug    ACC ≥ 0.91", c2_cls_s['accuracy']['mean'] >= 0.91),
    ("C3 Scaffold NoAug ACC ≥ 0.81", c3_cls_s['accuracy']['mean'] >= 0.81),
    ("C4 Scaffold Aug   ACC ≥ 0.87", c4_cls_s['accuracy']['mean'] >= 0.87),
    ("C2 vs C4 gap ≤ 0.04",
     abs(c2_cls_s['accuracy']['mean']-c4_cls_s['accuracy']['mean']) <= 0.04),
    ("C2 R² ≥ 0.75", c2_reg_s['r2']['mean'] >= 0.75),
    ("C4 R² ≥ 0.75", c4_reg_s['r2']['mean'] >= 0.75),
]
for desc, passed in checks:
    print(f"  {'✅' if passed else '❌'}  {desc}")
 
print(f"\n  Figures saved to: {FIG_DIR}")
print("\nEXPERIMENT COMPLETE!")
 


  FINAL ACCURACY & REGRESSION SUMMARY

  C1 Random  | No Augmentation
    ACC    = 0.8835 ± 0.0083
    AUC    = 0.9469 ± 0.0057
    MCC    = 0.7686 ± 0.0168
    Sens   = 0.9107 ± 0.0192
    Spec   = 0.8563 ± 0.0206
    F1     = 0.8866 ± 0.0082
    R²     = 0.6824 ± 0.0144
    RMSE   = 0.6848 ± 0.0142
    MAE    = 0.5034 ± 0.0099

  C2 Random  | With Augmentation
    ACC    = 0.8893 ± 0.0070
    AUC    = 0.9512 ± 0.0042
    MCC    = 0.7799 ± 0.0141
    Sens   = 0.8763 ± 0.0232
    Spec   = 0.9023 ± 0.0290
    F1     = 0.8878 ± 0.0065
    R²     = 0.7157 ± 0.0227
    RMSE   = 0.6474 ± 0.0236
    MAE    = 0.4566 ± 0.0113

  C3 Scaffold| No Augmentation
    ACC    = 0.8527 ± 0.0115
    AUC    = 0.9136 ± 0.0104
    MCC    = 0.7047 ± 0.0226
    Sens   = 0.8693 ± 0.0307
    Spec   = 0.8334 ± 0.0196
    F1     = 0.8539 ± 0.0193
    R²     = 0.5735 ± 0.0240
    RMSE   = 0.7858 ± 0.0380
    MAE    = 0.5755 ± 0.0276

  C4 Scaffold| With Augmentation
    ACC    = 0.8672 ± 0.0133
    AUC    = 0.92

In [44]:
# =============================================================================
# VISUALIZATIONS
# =============================================================================
 
CONDITIONS = ["C1\nRandom\nNoAug","C2\nRandom\nAug",
              "C3\nScaffold\nNoAug","C4\nScaffold\nAug"]
cond_names = ["C1: Random | No Aug","C2: Random | Aug",
              "C3: Scaffold | No Aug","C4: Scaffold | Aug"]
ALL_CLS_S = [c1_cls_s,c2_cls_s,c3_cls_s,c4_cls_s]
ALL_REG_S = [c1_reg_s,c2_reg_s,c3_reg_s,c4_reg_s]
ALL_PL    = [c1_pl,c2_pl,c3_pl,c4_pl]
ALL_CLS   = [c1_cls,c2_cls,c3_cls,c4_cls]
ALL_SIM   = [c1_sim,c2_sim,c3_sim,c4_sim]
 
# FIG 2
metrics_cls = ['accuracy','auc','mcc','sensitivity','specificity','f1',
               'precision','balanced_accuracy']
fig, axes = plt.subplots(2,4,figsize=(18,8)); axes = axes.flatten()
for ax, m in zip(axes, metrics_cls):
    means=[s[m]['mean'] for s in ALL_CLS_S]; stds=[s[m]['std'] for s in ALL_CLS_S]
    bars=ax.bar(CONDITIONS,means,color=COLORS,alpha=0.85,edgecolor='white',linewidth=0.8)
    ax.errorbar(range(4),means,yerr=stds,fmt='none',color='black',capsize=4,linewidth=1.5)
    ax.set_title(m.replace('_',' ').title(),fontweight='bold',fontsize=11)
    ax.set_ylim(0,1.08); ax.set_ylabel("Score"); ax.tick_params(axis='x',labelsize=8)
    for bar,mean in zip(bars,means):
        ax.text(bar.get_x()+bar.get_width()/2,mean+0.01,f'{mean:.3f}',
                ha='center',va='bottom',fontsize=8,fontweight='bold')
plt.suptitle("Classification Metrics — All Four Conditions",fontweight='bold',fontsize=13,y=1.02)
plt.tight_layout(); savefig("02_classification_metrics.png")
 
# FIG 3
fig,axes=plt.subplots(1,3,figsize=(14,5))
for ax,m in zip(axes,['r2','rmse','mae']):
    means=[s[m]['mean'] for s in ALL_REG_S]; stds=[s[m]['std'] for s in ALL_REG_S]
    bars=ax.bar(CONDITIONS,means,color=COLORS,alpha=0.85,edgecolor='white',linewidth=0.8)
    ax.errorbar(range(4),means,yerr=stds,fmt='none',color='black',capsize=4,linewidth=1.5)
    ax.set_title(m.upper(),fontweight='bold',fontsize=12); ax.set_ylabel("Score")
    ax.tick_params(axis='x',labelsize=8)
    for bar,mean in zip(bars,means):
        ax.text(bar.get_x()+bar.get_width()/2,mean+0.005,f'{mean:.3f}',
                ha='center',va='bottom',fontsize=9,fontweight='bold')
    if m=='r2':
        ax.axhline(0.75,color='red',linestyle='--',linewidth=1.5,label='R²=0.75 target')
        ax.legend(fontsize=9)
plt.suptitle("Regression Metrics — All Conditions",fontweight='bold',fontsize=13)
plt.tight_layout(); savefig("03_regression_metrics.png")
 
# FIG 4
fig,axes=plt.subplots(2,2,figsize=(13,11))
for ax,pl,cname,col in zip(axes.flatten(),ALL_PL,cond_names,COLORS):
    mean_fpr=np.linspace(0,1,200); tprs,aucs=[],[]
    for probs,labels,_,_ in pl:
        fpr,tpr,_=roc_curve(labels,probs); aucs.append(roc_auc_score(labels,probs))
        ax.plot(fpr,tpr,alpha=0.35,color=col,linewidth=1.2)
        tprs.append(np.interp(mean_fpr,fpr,tpr))
    mean_tpr=np.mean(tprs,axis=0); mean_tpr[0]=0.0; mean_tpr[-1]=1.0
    std_tpr=np.std(tprs,axis=0)
    ax.plot(mean_fpr,mean_tpr,color=col,linewidth=2.5,
            label=f"Mean AUC={np.mean(aucs):.4f}±{np.std(aucs):.4f}")
    ax.fill_between(mean_fpr,np.clip(mean_tpr-std_tpr,0,1),
                    np.clip(mean_tpr+std_tpr,0,1),alpha=0.18,color=col,label="±1 SD")
    ax.plot([0,1],[0,1],'k--',linewidth=1,alpha=0.5)
    ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(cname,fontweight='bold'); ax.legend(loc='lower right',fontsize=9)
plt.suptitle("ROC Curves — 5 Folds per Condition",fontweight='bold',fontsize=13)
plt.tight_layout(); savefig("04_auroc_curves.png")
 
# FIG 5
fig,ax=plt.subplots(figsize=(8,7))
for pl,cname,col in zip(ALL_PL,cond_names,COLORS):
    mean_fpr=np.linspace(0,1,200); tprs,aucs=[],[]
    for probs,labels,_,_ in pl:
        fpr,tpr,_=roc_curve(labels,probs); aucs.append(roc_auc_score(labels,probs))
        tprs.append(np.interp(mean_fpr,fpr,tpr))
    ax.plot(mean_fpr,np.mean(tprs,axis=0),color=col,linewidth=2.2,
            label=f"{cname.split(':')[0]} (AUC={np.mean(aucs):.4f})")
ax.plot([0,1],[0,1],'k--',linewidth=1,alpha=0.5,label='Random')
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("AUROC — All Conditions Overlaid",fontweight='bold')
ax.legend(loc='lower right',fontsize=10); plt.tight_layout(); savefig("05_auroc_overlay.png")
 
# FIG 6
fig,axes=plt.subplots(2,2,figsize=(13,11))
for ax,pl,cname,col in zip(axes.flatten(),ALL_PL,cond_names,COLORS):
    r2s=[r2_score(tru,pred) for _,_,tru,pred in pl]
    best_fold=int(np.argmax(r2s)); _,_,true_regs,avg_regs=pl[best_fold]
    r2=r2_score(true_regs,avg_regs); rmse=float(np.sqrt(mean_squared_error(true_regs,avg_regs)))
    ax.scatter(true_regs,avg_regs,alpha=0.45,s=16,color=col)
    lims=[min(true_regs.min(),avg_regs.min())-0.3,max(true_regs.max(),avg_regs.max())+0.3]
    ax.plot(lims,lims,'k--',linewidth=1.5,alpha=0.7)
    m_,b_=np.polyfit(true_regs,avg_regs,1); x_line=np.linspace(lims[0],lims[1],200)
    ax.plot(x_line,m_*x_line+b_,color=col,linewidth=2,alpha=0.8)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("True pIC50"); ax.set_ylabel("Predicted pIC50")
    ax.set_title(f"{cname}\nR²={r2:.4f}  RMSE={rmse:.4f}",fontweight='bold')
    ax.text(0.05,0.93,f"Best fold ({best_fold+1}/5)",transform=ax.transAxes,
            fontsize=9,color='gray')
plt.suptitle("Predicted vs True pIC50",fontweight='bold',fontsize=13)
plt.tight_layout(); savefig("06_regression_scatter.png")
 
# FIG 7
fig,axes=plt.subplots(1,4,figsize=(18,4))
for ax,pl,cname,col in zip(axes,ALL_PL,cond_names,COLORS):
    cms=[]
    for probs,labels,_,_ in pl:
        thrs=np.linspace(0.1,0.9,161)
        accs=[accuracy_score(labels,(probs>=t).astype(int)) for t in thrs]
        thr=float(thrs[np.argmax(accs)])
        cms.append(confusion_matrix(labels,(probs>=thr).astype(int)))
    mean_cm=np.mean(cms,axis=0)
    sns.heatmap(mean_cm,annot=True,fmt='.0f',ax=ax,
                cmap=sns.light_palette(col,as_cmap=True),
                xticklabels=['Pred 0','Pred 1'],yticklabels=['True 0','True 1'],
                linewidths=0.5,linecolor='gray')
    ax.set_title(cname.replace(":"," -"),fontweight='bold',fontsize=10)
plt.suptitle("Confusion Matrices — Mean over 5 Folds",fontweight='bold',fontsize=13)
plt.tight_layout(); savefig("07_confusion_matrices.png")
 
# FIG 8
fig,axes=plt.subplots(1,2,figsize=(13,5))
for ax,metric,title in zip(axes,['accuracy','auc'],['Accuracy','AUC-ROC']):
    data_v=[]
    for cname,cls_list in zip(cond_names,ALL_CLS):
        for r in cls_list: data_v.append({'Condition':cname,'Value':r[metric]})
    df_v=pd.DataFrame(data_v)
    sns.violinplot(data=df_v,x='Condition',y='Value',palette=COLORS,
                   inner='point',ax=ax,cut=0,scale='width')
    ax.set_title(title,fontweight='bold'); ax.set_ylabel(title); ax.set_xlabel("")
plt.suptitle("Distribution of Fold-Level ACC and AUC",fontweight='bold',fontsize=13)
plt.tight_layout(); savefig("08_violin_acc_auc.png")
 
# FIG 9
fig,axes=plt.subplots(1,2,figsize=(11,5))
for ax,noaug_s,aug_s,title in zip(axes,[c1_cls_s,c3_cls_s],[c2_cls_s,c4_cls_s],
    ['Random: Augmentation Effect','Scaffold: Augmentation Effect']):
    metrics_shown=['accuracy','auc','mcc','f1','balanced_accuracy']
    deltas=[aug_s[m]['mean']-noaug_s[m]['mean'] for m in metrics_shown]
    colors_bar=['#2ecc71' if d>=0 else '#e74c3c' for d in deltas]
    ax.barh(metrics_shown,deltas,color=colors_bar,edgecolor='white',alpha=0.85)
    ax.axvline(0,color='black',linewidth=1); ax.set_xlabel("Δ (Aug − NoAug)")
    ax.set_title(title,fontweight='bold')
    for i,d in enumerate(deltas):
        ax.text(d+(0.001 if d>=0 else -0.001),i,f'{d:+.4f}',va='center',
                ha='left' if d>=0 else 'right',fontsize=9)
plt.suptitle("Effect of Data Augmentation",fontweight='bold',fontsize=12)
plt.tight_layout(); savefig("09_augmentation_delta.png")
 
# FIG 10
fig,axes=plt.subplots(1,2,figsize=(11,5))
for ax,aug_label,rand_s,scaf_s in zip(axes,['No Augmentation','With Augmentation'],
                                        [c1_cls_s,c2_cls_s],[c3_cls_s,c4_cls_s]):
    metrics_shown=['accuracy','auc','mcc','f1']; x=np.arange(4); w=0.35
    ax.bar(x-w/2,[rand_s[m]['mean'] for m in metrics_shown],w,label='Random',
           color='#4C72B0',alpha=0.85,edgecolor='white')
    ax.bar(x+w/2,[scaf_s[m]['mean'] for m in metrics_shown],w,label='Scaffold',
           color='#C44E52',alpha=0.85,edgecolor='white')
    ax.set_xticks(x); ax.set_xticklabels(metrics_shown)
    ax.set_ylim(0,1.08); ax.set_ylabel("Score"); ax.set_title(aug_label,fontweight='bold')
    ax.legend()
    ax.errorbar(x-w/2,[rand_s[m]['mean'] for m in metrics_shown],
                yerr=[rand_s[m]['std'] for m in metrics_shown],
                fmt='none',color='black',capsize=3,linewidth=1.2)
    ax.errorbar(x+w/2,[scaf_s[m]['mean'] for m in metrics_shown],
                yerr=[scaf_s[m]['std'] for m in metrics_shown],
                fmt='none',color='black',capsize=3,linewidth=1.2)
plt.suptitle("Random vs Scaffold Split",fontweight='bold',fontsize=12)
plt.tight_layout(); savefig("10_random_vs_scaffold_gap.png")
 
# FIG 11
r2_matrix=np.array([[r2_score(true,pred) for _,_,true,pred in pl] for pl in ALL_PL])
fig,ax=plt.subplots(figsize=(9,4))
sns.heatmap(r2_matrix,annot=True,fmt='.4f',ax=ax,cmap='YlGn',
            xticklabels=[f'Fold {i+1}' for i in range(5)],yticklabels=cond_names,
            linewidths=0.5,linecolor='gray',vmin=0.5,vmax=1.0)
ax.set_title("R² per Fold — All Conditions",fontweight='bold')
plt.tight_layout(); savefig("11_r2_heatmap.png")
 
# FIG 12
fig,ax=plt.subplots(figsize=(8,5))
for pl,sim_list,cname,col,cls_list in zip(ALL_PL,ALL_SIM,cond_names,COLORS,ALL_CLS):
    ax.scatter([s[0] for s in sim_list],[r['accuracy'] for r in cls_list],
               color=col,label=cname,s=70,alpha=0.75,edgecolors='white',linewidths=0.5)
ax.set_xlabel("Mean Max Tanimoto Similarity"); ax.set_ylabel("Fold Accuracy")
ax.set_title("Chemical Similarity vs Fold Accuracy",fontweight='bold')
ax.legend(fontsize=9); plt.tight_layout(); savefig("12_tanimoto_vs_acc.png")

  [saved] .\figures\02_classification_metrics.png
  [saved] .\figures\03_regression_metrics.png
  [saved] .\figures\04_auroc_curves.png
  [saved] .\figures\05_auroc_overlay.png
  [saved] .\figures\06_regression_scatter.png
  [saved] .\figures\07_confusion_matrices.png
  [saved] .\figures\08_violin_acc_auc.png
  [saved] .\figures\09_augmentation_delta.png
  [saved] .\figures\10_random_vs_scaffold_gap.png
  [saved] .\figures\11_r2_heatmap.png
  [saved] .\figures\12_tanimoto_vs_acc.png


In [45]:
# =============================================================================
# FINAL PRINTED SUMMARY
# =============================================================================
print("\n" + "="*80)
print("  FINAL ACCURACY & REGRESSION SUMMARY (Mean ± SD)")
print("="*80)
for cname, cs, rs in zip(
        ["C1 Random  | No Augmentation",
         "C2 Random  | With Augmentation",
         "C3 Scaffold| No Augmentation",
         "C4 Scaffold| With Augmentation"],
        [c1_cls_s, c2_cls_s, c3_cls_s, c4_cls_s],
        [c1_reg_s, c2_reg_s, c3_reg_s, c4_reg_s]):
    print(f"\n  {cname}")
    print(f"    ACC  = {cs['accuracy']['mean']:.4f} ± {cs['accuracy']['std']:.4f}")
    print(f"    AUC  = {cs['auc']['mean']:.4f} ± {cs['auc']['std']:.4f}")
    print(f"    MCC  = {cs['mcc']['mean']:.4f} ± {cs['mcc']['std']:.4f}")
    print(f"    Sens = {cs['sensitivity']['mean']:.4f} ± {cs['sensitivity']['std']:.4f}")
    print(f"    Spec = {cs['specificity']['mean']:.4f} ± {cs['specificity']['std']:.4f}")
    print(f"    F1   = {cs['f1']['mean']:.4f} ± {cs['f1']['std']:.4f}")
    print(f"    R²   = {rs['r2']['mean']:.4f} ± {rs['r2']['std']:.4f}")
    print(f"    RMSE = {rs['rmse']['mean']:.4f} ± {rs['rmse']['std']:.4f}")
    print(f"    MAE  = {rs['mae']['mean']:.4f} ± {rs['mae']['std']:.4f}")
 
# Target checks
print("\n" + "="*80)
print("  TARGET CHECKS")
print("="*80)
checks = [
    ("C1 Random NoAug  ACC ≥ 0.85", c1_cls_s['accuracy']['mean'] >= 0.85),
    ("C2 Random Aug    ACC ≥ 0.91", c2_cls_s['accuracy']['mean'] >= 0.91),
    ("C3 Scaffold NoAug ACC ≥ 0.81", c3_cls_s['accuracy']['mean'] >= 0.81),
    ("C4 Scaffold Aug  ACC ≥ 0.87", c4_cls_s['accuracy']['mean'] >= 0.87),
    ("C2 vs C4 gap ≤ 0.04",
     abs(c2_cls_s['accuracy']['mean'] - c4_cls_s['accuracy']['mean']) <= 0.04),
    ("C2 R² ≥ 0.75",  c2_reg_s['r2']['mean'] >= 0.75),
    ("C4 R² ≥ 0.75",  c4_reg_s['r2']['mean'] >= 0.75),
]
for desc, passed in checks:
    icon = "✅" if passed else "❌"
    print(f"  {icon}  {desc}")
 
print("\n  All figures saved to:", FIG_DIR)
print("\nEXPERIMENT COMPLETE!")
 


  FINAL ACCURACY & REGRESSION SUMMARY (Mean ± SD)

  C1 Random  | No Augmentation
    ACC  = 0.8835 ± 0.0083
    AUC  = 0.9469 ± 0.0057
    MCC  = 0.7686 ± 0.0168
    Sens = 0.9107 ± 0.0192
    Spec = 0.8563 ± 0.0206
    F1   = 0.8866 ± 0.0082
    R²   = 0.6824 ± 0.0144
    RMSE = 0.6848 ± 0.0142
    MAE  = 0.5034 ± 0.0099

  C2 Random  | With Augmentation
    ACC  = 0.8893 ± 0.0070
    AUC  = 0.9512 ± 0.0042
    MCC  = 0.7799 ± 0.0141
    Sens = 0.8763 ± 0.0232
    Spec = 0.9023 ± 0.0290
    F1   = 0.8878 ± 0.0065
    R²   = 0.7157 ± 0.0227
    RMSE = 0.6474 ± 0.0236
    MAE  = 0.4566 ± 0.0113

  C3 Scaffold| No Augmentation
    ACC  = 0.8527 ± 0.0115
    AUC  = 0.9136 ± 0.0104
    MCC  = 0.7047 ± 0.0226
    Sens = 0.8693 ± 0.0307
    Spec = 0.8334 ± 0.0196
    F1   = 0.8539 ± 0.0193
    R²   = 0.5735 ± 0.0240
    RMSE = 0.7858 ± 0.0380
    MAE  = 0.5755 ± 0.0276

  C4 Scaffold| With Augmentation
    ACC  = 0.8672 ± 0.0133
    AUC  = 0.9282 ± 0.0107
    MCC  = 0.7353 ± 0.0261
    Sen